### Processing the data

#### data wrangling and cleaning

In [ ]:
# import dependencies
import pandas as pd

# data cleaning process
df = pd.read_csv('resources/exoplanet_data.csv')
# df.head()

# drop all null columns and rows
column_clean = df.dropna(axis='columns', how='all')
# column_clean.head()
df_clean = column_clean.dropna()
df_clean.head(9)

#### pre-processing data using sklearn

In [ ]:
from sklearn.model_selection import train_test_split
# create x and y variables
X = df_clean.drop('koi_disposition', axis=1)
y = df_clean['koi_disposition']

# split data to create training and testing data
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y)

from sklearn.preprocessing import MinMaxScaler
# scale x values
X_scaler = MinMaxScaler().fit(X_train)
X_train_scaled = X_scaler.transform(X_train)
X_test_scaled = X_scaler.transform(X_test)

### Model building

#### model training using LR

In [ ]:
# start training the datasets
from sklearn.linear_model import LogisticRegression
classifier = LogisticRegression(solver='newton-cg', multi_class='auto')
classifier

In [ ]:
classifier.fit(X_train_scaled, y_train)

In [ ]:
print(f'training data = {classifier.score(X_train_scaled, y_train)}')
print(f'testing data = {classifier.score(X_test_scaled, y_test)}')

### Feature selection
##### this project will use recursive feature elimination to assign weights to features.

In [ ]:
# read https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.RFECV.html
# feature selection used in this project is RFECV
from sklearn.feature_selection import RFECV
# we will use RFECV feature selection 
fn = X.columns.tolist()
feature_select=RFECV(estimator=classifier, cv=5, step=1)
selected_features=feature_select.fit(X_train_scaled, y_train)

In [ ]:
# beast feature selection processure_select.support_
feature_searching = sorted(zip(feature_select.ranking_, fn, feature_select.support_))
# feature_searching
features_sorted = pd.DataFrame(feature_searching, columns=['rank', 'feature', 'support']).set_index('feature')
# features_sorted.max()
features_sorted.head()

In [ ]:
# eliminate all features with ranking of 2 and above
approved_features = []
for i in feature_searching:
    if i[0] < 12:
        approved_features.append(i[1])


In [ ]:
# use the new data for the entire model
from sklearn.preprocessing import MinMaxScaler

X_train_new = X_train[approved_features]
X_test_new = X_test[approved_features]
# X_test_new

scaler = MinMaxScaler()
X_scaler=scaler.fit(X_train_new)

X_train_scaled = X_scaler.transform(X_train_new)
X_test_scaled = X_scaler.transform(X_test_new)

new_model = LogisticRegression(solver='newton-cg', multi_class='auto')
new_model.fit(X_train_scaled, y_train)

print(f'training data = {new_model.score(X_train_scaled, y_train)}')
print(f'testing data = {new_model.score(X_test_scaled, y_test)}')


### Model tuning process

In [ ]:
# create gridsearch estimator along with a parameter object having the values to adjust
from sklearn.model_selection import GridSearchCV
import numpy as np

model_2=LogisticRegression(solver='newton-cg', multi_class='auto')
param_grid = {'C': np.logspace(0, 4, 10),
                'penalty': ['l2']
                }
grid=GridSearchCV(model_2, param_grid, verbose=0, cv=5)

In [ ]:
# fit model using gridsearch estimator
fit_model=grid.fit(X_train_scaled, y_train)

In [ ]:
# train the tuned model
# select tuned params
c_value = grid.best_params_['C']
penalty_value = grid.best_params_['penalty']

# model tuned
t_model = LogisticRegression(solver='newton-cg', multi_class='auto', C = c_value, penalty=penalty_value)
t_model.fit(X_train_scaled, y_train)

print(f'training data_tuned = {t_model.score(X_train_scaled, y_train)}')
print(f'testing data_tuned = {t_model.score(X_test_scaled, y_test)}')

### Model prediction

In [ ]:
# predict using the trained model
predictions = t_model.predict(X_test_scaled)

print('test accuracy: %.3f'%grid.score(X_test_scaled, y_test))

sorting_y_test_data = y_test.unique().tolist()

overall = {'real': y_test, 'predictions':predictions}

# create a dataframe for the predicted and actual values to compare against
comparison_df = pd.DataFrame(overall).set_index('real')
comparison_df.reset_index().head(9)

In [ ]:
# evaluate the model
# make evaluations of the models
a=round(classifier.score(X_test_scaled, y_test)*100,2)
eval = {'model tested': ['root model', 'selected features model', 'tuned model'],
    'accuracy': [
    f'{round(classifier.score(X_test_scaled, y_test)*100,2)}%', 
    f'{round(new_model.score(X_test_scaled, y_test)*100,2)}%', 
    f'{round(t_model.score(X_test_scaled, y_test)*100,2)}%']}

eval_df = pd.DataFrame(eval).set_index('model tested').reset_index()
eval_df

In [ ]:
# save the model scores to a csv 
eval_df.to_csv('resources/LR_scores.csv')


import joblib
filename = 'ML_models/LR.sav'
joblib.dump(t_model, filename)
